# 하이브리드 모핑 파이프라인 — Colab 실행 노트북

로컬에서 만든 카메라 궤적(JSON)을 레포에서 받아, Blender로 depth/normal pass를 렌더하고,
ComfyUI로 실제 영상 프레임을 생성한 뒤, 디테일 복원 후 mp4로 출력한다.

**런타임: GPU (T4 이상) 필요** — 상단 메뉴 '런타임 > 런타임 유형 변경'에서 확인.

실행 순서: 0(클론) → 1(Blender) → 2(ComfyUI 설치) → 2b(모델 가중치 받기) → 3(워크플로우 실행) → 4(디테일 복원) → 5(출력)

In [ ]:
# @title 0. 레포 클론 + 기본 설정
REPO_URL = "https://github.com/<YOUR_GITHUB_USERNAME>/stereo-morph-pipeline.git"  # @param {type:"string"}
SHOT_ID = "demo_bullet_time"  # @param {type:"string"}

import os
%cd /content
!rm -rf stereo-morph-pipeline
!git clone $REPO_URL
%cd /content/stereo-morph-pipeline
!pip install -q -r requirements.txt

In [ ]:
# @title 1. Blender 설치 + 더미 메시 생성 + 카메라 패스 적용(depth/normal 렌더)
# Colab 기본 이미지엔 Blender가 없음 — 헤드리스 바이너리를 받아 설치
!wget -q https://download.blender.org/release/Blender4.2/blender-4.2.0-linux-x64.tar.xz -O /tmp/blender.tar.xz
!tar -xf /tmp/blender.tar.xz -C /opt
!ln -sf /opt/blender-4.2.0-linux-x64/blender /usr/local/bin/blender
!blender --version

SHOT_JSON = f"/content/stereo-morph-pipeline/control/presets/{SHOT_ID}.json"
MESH_PATH = f"/content/dummy_subject_{SHOT_ID}.glb"
RENDER_DIR = f"/content/render/{SHOT_ID}"

!blender -b -P blender/generate_dummy_mesh.py -- --out $MESH_PATH
!blender -b -P blender/apply_camera_path.py -- --path $SHOT_JSON --mesh $MESH_PATH --out-dir $RENDER_DIR
print("depth/normal pass:", RENDER_DIR)

In [ ]:
# @title 2. ComfyUI + 커스텀 노드팩 설치
%cd /content
!git clone --depth 1 https://github.com/comfyanonymous/ComfyUI.git
%cd /content/ComfyUI
!pip install -q -r requirements.txt

%cd /content/ComfyUI/custom_nodes
!git clone --depth 1 https://github.com/Kosinkadink/ComfyUI-AnimateDiff-Evolved.git
!git clone --depth 1 https://github.com/Kosinkadink/ComfyUI-VideoHelperSuite.git

# 체크포인트/컨트롤넷/모션모듈 가중치는 용량이 커서 Drive 마운트 후 심링크 권장.
# 최초 1회 Google Drive에 받아두고, 이후 세션부터는 다운로드 생략.
from google.colab import drive
drive.mount('/content/drive')
# TODO: /content/drive/MyDrive/morph-pipeline-models/ 아래에 체크포인트 배치 후
# ComfyUI/models/{checkpoints,controlnet,animatediff_models} 로 심링크

In [ ]:
# @title 2b. 모델 가중치 받기 (체크포인트/ControlNet/AnimateDiff 모션모듈) - 최초 1회만 느림, 이후 Drive 캐시로 빠름
import os

DRIVE_CACHE = "/content/drive/MyDrive/morph-pipeline-models"
os.makedirs(DRIVE_CACHE, exist_ok=True)

MODELS = [
    ("checkpoints", "v1-5-pruned-emaonly.safetensors",
     "https://huggingface.co/stable-diffusion-v1-5/stable-diffusion-v1-5/resolve/main/v1-5-pruned-emaonly.safetensors"),
    ("controlnet", "control_v11f1p_sd15_depth.pth",
     "https://huggingface.co/lllyasviel/ControlNet-v1-1/resolve/main/control_v11f1p_sd15_depth.pth"),
    ("animatediff_models", "mm_sd_v15_v2.ckpt",
     "https://huggingface.co/guoyww/animatediff/resolve/main/mm_sd_v15_v2.ckpt"),
]

for subdir, filename, url in MODELS:
    target_dir = f"/content/ComfyUI/models/{subdir}"
    os.makedirs(target_dir, exist_ok=True)
    target_path = f"{target_dir}/{filename}"
    cache_path = f"{DRIVE_CACHE}/{filename}"

    if os.path.exists(target_path):
        print(f"이미 있음: {target_path}")
    elif os.path.exists(cache_path):
        print(f"Drive 캐시에서 복사: {filename}")
        !cp "{cache_path}" "{target_path}"
    else:
        print(f"다운로드: {filename}")
        !wget -q --show-progress -O "{target_path}" "{url}"
        print(f"Drive에 캐시 저장: {filename}")
        !cp "{target_path}" "{cache_path}"

# AnimateDiff-Evolved는 모션모듈을 custom_nodes 쪽 models/ 폴더도 봅니다 - 심링크로 양쪽 커버
ade_models_dir = "/content/ComfyUI/custom_nodes/ComfyUI-AnimateDiff-Evolved/models"
os.makedirs(ade_models_dir, exist_ok=True)
!ln -sf /content/ComfyUI/models/animatediff_models/mm_sd_v15_v2.ckpt "{ade_models_dir}/mm_sd_v15_v2.ckpt"

print()
print("모델 준비 완료 (체크포인트/ControlNet/모션모듈 전부 SD1.5 계열로 통일)")

In [ ]:
# @title (검증) 설치된 노드 class_type 확인 — workflow_builder.py 와 이름이 다르면 여기서 맞춰야 함
import subprocess, threading, time, requests

def start_comfyui():
    subprocess.Popen(["python", "main.py", "--listen", "0.0.0.0", "--port", "8188"], cwd="/content/ComfyUI")

threading.Thread(target=start_comfyui, daemon=True).start()
time.sleep(15)

resp = requests.get("http://127.0.0.1:8188/object_info").json()
needed = ["VHS_LoadImagesPath", "ADE_AnimateDiffLoaderGen1", "VHS_VideoCombine", "ControlNetApplyAdvanced"]
for n in needed:
    print(n, "OK" if n in resp else "⚠️ 이름이 다름 — comfyui/workflow_builder.py 수정 필요")

In [ ]:
# @title 3. 워크플로우 조립 + 큐 전송
import sys, json, requests
sys.path.insert(0, "/content/stereo-morph-pipeline/comfyui")
from workflow_builder import build_workflow

SUBJECT_IMAGE = "/content/subject_ref.png"  # 업로드한 피사체 참조 이미지로 교체

wf = build_workflow(
    shot_id=SHOT_ID,
    depth_pass_dir=f"{RENDER_DIR}/depth",
    subject_image_path=SUBJECT_IMAGE,
    positive_prompt="cinematic photoreal subject, bullet time orbit shot, consistent identity, sharp focus",
)
r = requests.post("http://127.0.0.1:8188/prompt", json={"prompt": wf})
print(r.status_code, r.text[:500])

In [ ]:
# @title 4. 디테일 복원 (옵션 — 필요한 단계만 레포 경로 채우고 실행)
sys.path.insert(0, "/content/stereo-morph-pipeline/refinement")
from refine import refine_chain

RAW_FRAMES_DIR = "/content/ComfyUI/output"  # ComfyUI가 뱉은 생성 프레임 폴더로 수정

refine_chain(
    raw_frames_dir=RAW_FRAMES_DIR,
    work_dir=f"/content/refine_work/{SHOT_ID}",
    final_out_path=f"/content/drive/MyDrive/morph-pipeline-output/{SHOT_ID}.mp4",
    fps=24,
    rife_repo_dir=None,        # 필요 시 ECCV2022-RIFE 클론 경로
    codeformer_repo_dir=None,  # 필요 시 CodeFormer 클론 경로
    realesrgan_repo_dir=None,  # 필요 시 Real-ESRGAN 클론 경로
)

## 5. 출력
결과 mp4는 `Google Drive > morph-pipeline-output/`에 저장됨. 로컬 PC에서는 Drive 동기화로
받거나, `output/` 폴더로 수동 복사.